# BondSpot dashboard — duration & debt composition over time

Wykresy:
1. **Portfolio-weighted metryki** (Mod/Mac Duration, ATM, ATR) w czasie
2. **Per typ obligacji** (DS/PS/WS/WZ/IZ/...) — facet 2×2
3. **Skład długu** — stacked area (bondy + bony skarbowe)

## Setup
1. `pip install -r ../requirements-notebook.txt`
2. Utwórz `.env` w roocie repo z `SUPABASE_URL` i `SUPABASE_SERVICE_ROLE_KEY`
3. Run all cells

In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import requests
from dotenv import load_dotenv
from plotly.subplots import make_subplots

# Renderer wymagany zeby fig.show() emitowal HTML/JS przy nbconvert -> HTML;
# live Jupyter zwykle uzywa 'plotly_mimetype' ale to nie renderuje w GH Pages.
pio.renderers.default = "notebook_connected"

load_dotenv(Path("..") / ".env")

SUPABASE_URL = os.environ["SUPABASE_URL"].rstrip("/")
SUPABASE_KEY = os.environ["SUPABASE_SERVICE_ROLE_KEY"]

HEADERS = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}",
    "Content-Type": "application/json",
}

PAGE_SIZE = 1000  # Supabase PGRST_DB_MAX_ROWS default - kapuje response do 1000


def _paginate(method, url, *, json_body=None, timeout=120, max_pages=200):
    """Stronicowane wywolanie endpoint-u Supabase. PostgREST tnie response do
    PGRST_DB_MAX_ROWS (1000 default na Supabase) niezaleznie od Range header,
    wiec idziemy w petli inkrementujac offset i konkatenujac wyniki."""
    rows: list = []
    for page in range(max_pages):
        offset = page * PAGE_SIZE
        h = {**HEADERS,
             "Range-Unit": "items",
             "Range": f"{offset}-{offset + PAGE_SIZE - 1}"}
        kw = {"headers": h, "timeout": timeout}
        if json_body is not None:
            kw["json"] = json_body
        r = requests.request(method, url, **kw)
        if r.status_code not in (200, 206):
            r.raise_for_status()
        chunk = r.json()
        if not isinstance(chunk, list):
            return chunk  # scalar / dict response - return as-is
        rows.extend(chunk)
        if len(chunk) < PAGE_SIZE:
            return rows
    raise RuntimeError(f"_paginate hit max_pages={max_pages} without finishing")


def rpc(name: str, payload: dict | None = None) -> pd.DataFrame:
    rows = _paginate("POST", f"{SUPABASE_URL}/rest/v1/rpc/{name}",
                     json_body=payload or {})
    return pd.DataFrame(rows)


def fetch_view(name: str, query: str = "?select=*") -> pd.DataFrame:
    rows = _paginate("GET", f"{SUPABASE_URL}/rest/v1/{name}{query}")
    return pd.DataFrame(rows)


print("Connected.")

## 1. Portfolio-weighted metryki w czasie

Z `v_portfolio_metrics_daily` (ważone outstanding-em dziennym, bondy hurtowe).

In [ ]:
df1 = fetch_view("v_portfolio_metrics_daily", "?select=*&order=fixing_date.asc")
df1["fixing_date"] = pd.to_datetime(df1["fixing_date"])
for c in ["portfolio_mod_duration", "portfolio_mac_duration", "portfolio_atm",
         "portfolio_atr", "portfolio_yield_pct", "total_outstanding_mln_pln"]:
    if c in df1.columns:
        df1[c] = pd.to_numeric(df1[c], errors="coerce")

print(f"Days: {len(df1)},  range: {df1.fixing_date.min().date()} → {df1.fixing_date.max().date()}")
df1.tail()

In [ ]:
fig = go.Figure()
for col, label in [
    ("portfolio_mod_duration", "Modified Duration"),
    ("portfolio_mac_duration", "Macaulay Duration"),
    ("portfolio_atm", "ATM (years to maturity)"),
    ("portfolio_atr", "ATR (years to refixing)"),
]:
    fig.add_trace(go.Scatter(x=df1["fixing_date"], y=df1[col], name=label, mode="lines"))

fig.update_layout(
    title="Portfolio-weighted metryki polskiego długu (bondy hurtowe)",
    xaxis_title="Data fixingu (EOD = sesja 2)",
    yaxis_title="Lata",
    hovermode="x unified",
    template="plotly_white",
    height=500,
    legend=dict(orientation="h", y=-0.15),
)
fig.show()

## 2. Per typ obligacji (DS, PS, WS, WZ, IZ, …)

Każdy z 4 paneli pokazuje jedną metrykę, kolory = typy obligacji.

Czego się spodziewać:
- **WS** (długie 20-30Y) — najwyższe Mac/Mod Duration
- **WZ** (floatery) — Mod/Mac ≈ ATR (≤ 0.5Y), bo resetują się co 6mc
- **OK** — wszystkie metryki = ATM (zerokuponowe), maleją liniowo do wykupu

In [ ]:
df2 = fetch_view("v_portfolio_metrics_by_type", "?select=*&order=fixing_date.asc,bond_type.asc")
df2["fixing_date"] = pd.to_datetime(df2["fixing_date"])
for c in ["total_mln_pln", "w_mod_duration", "w_mac_duration", "w_atm", "w_atr", "w_yield_pct"]:
    df2[c] = pd.to_numeric(df2[c], errors="coerce")

print(f"Rows: {len(df2)},  types: {sorted(df2.bond_type.unique())}")
df2.tail()

In [ ]:
metrics = [
    ("w_mod_duration", "Modified Duration (lata)"),
    ("w_mac_duration", "Macaulay Duration (lata)"),
    ("w_atm", "ATM (lata)"),
    ("w_atr", "ATR (lata)"),
]
fig = make_subplots(rows=2, cols=2, subplot_titles=[m[1] for m in metrics],
                    shared_xaxes=True, vertical_spacing=0.10, horizontal_spacing=0.08)

types_sorted = sorted(df2.bond_type.unique())
colors = px.colors.qualitative.Set2 + px.colors.qualitative.Set3
color_map = {bt: colors[i % len(colors)] for i, bt in enumerate(types_sorted)}

for i, (col, _) in enumerate(metrics):
    row, c = i // 2 + 1, i % 2 + 1
    for bt in types_sorted:
        sub = df2[df2.bond_type == bt].sort_values("fixing_date")
        fig.add_trace(
            go.Scatter(
                x=sub["fixing_date"], y=sub[col],
                name=bt, legendgroup=bt,
                showlegend=(i == 0), mode="lines",
                line=dict(color=color_map[bt], width=1.5),
            ),
            row=row, col=c,
        )

fig.update_layout(
    title="Metryki ważone outstanding per typ obligacji",
    template="plotly_white",
    height=750,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.05),
)
fig.show()

## 3. Skład długu — stacked area

Bondy hurtowe per typ + bony skarbowe (jako jeden kubełek `tbill`). Pokazuje jak ewoluowała struktura zadłużenia.

Resample do miesięcznych snapshotów (end-of-month), inaczej wykres jest zaszumiony przy 3000+ dni × 8 typów.

In [ ]:
START_DATE = "2012-01-01"
TODAY = pd.Timestamp.today().normalize()

# Event-driven sklad dlugu z MF (niezalezne od BondSpota - chwytamy NZ-tki
# od dnia pierwszej emisji w MF, nie od pierwszego BondSpotowego fixingu).
# Widoki v_bond_outstanding_by_type_events i v_tbill_outstanding_events
# robia cumulative sum delty per typ na poziomie SQL.

bond_events = fetch_view(
    "v_bond_outstanding_by_type_events",
    "?select=*&order=change_date.asc,bond_type.asc",
)
bond_events["change_date"] = pd.to_datetime(bond_events["change_date"])
bond_events["outstanding_mln_pln"] = pd.to_numeric(
    bond_events["outstanding_mln_pln"], errors="coerce"
)

tbill_events = fetch_view(
    "v_tbill_outstanding_events",
    "?select=*&order=change_date.asc",
)
tbill_events["change_date"] = pd.to_datetime(tbill_events["change_date"])
tbill_events["outstanding_mln_pln"] = pd.to_numeric(
    tbill_events["outstanding_mln_pln"], errors="coerce"
)
tbill_events["bond_type"] = "tbill"

# Polacz w jeden long DataFrame
df3 = pd.concat([
    bond_events.rename(columns={"change_date": "event_date"}),
    tbill_events.rename(columns={"change_date": "event_date"})[
        ["event_date", "bond_type", "outstanding_mln_pln"]
    ],
], ignore_index=True)

# Pivot do wide: rows = event_date, cols = bond_type, values = outstanding
piv = df3.pivot_table(index="event_date", columns="bond_type",
                     values="outstanding_mln_pln", aggfunc="sum")

# Daily upsample + ffill (sparse event-driven series -> daily continuous;
# wartosci stale miedzy eventami)
piv = piv.resample("D").ffill().fillna(0)

# Trim do TODAY (zadnych przyszlych synthetic redemption dates)
piv = piv[piv.index <= TODAY]
# Trim do START_DATE
piv = piv[piv.index >= START_DATE]

# Reduce do dni z faktycznymi zmianami (auction-event datapointy)
mask = (piv != piv.shift(1)).any(axis=1)
piv = piv[mask]

# Order: typy od najwiekszego sredniego outstanding
order = piv.mean().sort_values(ascending=False).index.tolist()
piv = piv[order]

print(f"Event dates: {len(piv)}, range: {piv.index.min().date()} → {piv.index.max().date()}")
print(f"Latest total: {piv.iloc[-1].sum() / 1000:.1f} bln PLN")
print(f"Latest per type: {piv.iloc[-1].to_dict()}")
print(f"Types: {order}")
piv.tail()

In [ ]:
fig = go.Figure()
for bt in order:
    fig.add_trace(go.Scatter(
        x=piv.index, y=piv[bt] / 1000.0,
        name=bt, mode="lines", stackgroup="one",
        hovertemplate="%{y:.1f} bln PLN<extra>" + bt + "</extra>",
    ))

fig.update_layout(
    title="Skład długu skarbowego (bondy hurtowe + bony) — auction events",
    xaxis_title="Data aukcji / odkupu",
    yaxis_title="Outstanding (bln PLN)",
    template="plotly_white",
    hovermode="x unified",
    height=600,
    legend=dict(orientation="h", y=-0.15),
)
fig.show()

## 4. Aukcje obligacji skarbowych

Dane z arkusza MF Operacje (per leg). Filtr: `type_tx='S' AND type_op IN ('AS','AU')` — aukcje sprzedażowe (primary + top-up); switche (AZ) i odkupy (AO) pominięte.

Sekcje:
- **4a** — statystyki dla CAŁEJ aukcji dziennej (wszystkie serie razem)
- **4b** — per typ kuponu (I = inflacja, OS = O+S zerokuponowe i stałe, Z = zmienne)
- **4c** — per LataDoWykupu (tenor 2/5/10/20/30 lat)
- **4d** — szczegółowa tabela ostatniej aukcji

### 4a. Statystyki per aukcja (cały dzień)

Każda aukcja = wszystkie sprzedane tego dnia serie sumarycznie. Górny wykres: B/C, dolny: tail + concession.

In [ ]:
df4a = fetch_view(
    "v_auction_day_totals",
    "?type_op=in.(AS,AU)&select=*&order=auction_date.asc",
)
df4a["auction_date"] = pd.to_datetime(df4a["auction_date"])
for c in ["bid_to_cover", "bid_to_offer", "w_yield_avg", "w_tail_bp",
          "w_concession_bp", "total_sold_mln", "total_demand_mln",
          "nc_share_demand"]:
    if c in df4a.columns:
        df4a[c] = pd.to_numeric(df4a[c], errors="coerce")

agg = df4a.groupby("auction_date", as_index=False).agg(
    total_sold_mln=("total_sold_mln", "sum"),
    total_demand_mln=("total_demand_mln", "sum"),
    w_yield_avg=("w_yield_avg", "mean"),
    w_tail_bp=("w_tail_bp", "mean"),
    w_concession_bp=("w_concession_bp", "mean"),
)
agg["bid_to_cover"] = agg["total_demand_mln"] / agg["total_sold_mln"]

print(f"Auction days: {len(agg)},  range: {agg.auction_date.min().date()} → {agg.auction_date.max().date()}")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                    subplot_titles=("Bid-to-cover + total sold", "Tail (bp) + concession (bp)"),
                    specs=[[{"secondary_y": True}], [{}]])

fig.add_trace(go.Bar(x=agg["auction_date"], y=agg["total_sold_mln"],
                     name="Total sold (mln PLN)", marker_color="lightgrey",
                     opacity=0.6), row=1, col=1, secondary_y=True)
fig.add_trace(go.Scatter(x=agg["auction_date"], y=agg["bid_to_cover"],
                         name="Bid-to-cover", mode="lines+markers",
                         line=dict(color="navy", width=2)), row=1, col=1, secondary_y=False)

fig.add_trace(go.Scatter(x=agg["auction_date"], y=agg["w_tail_bp"],
                         name="Tail (bp)", mode="lines+markers",
                         line=dict(color="firebrick", width=1.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=agg["auction_date"], y=agg["w_concession_bp"],
                         name="Concession (bp)", mode="lines+markers",
                         line=dict(color="seagreen", width=1.5)), row=2, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="grey", row=2, col=1)

fig.update_yaxes(title_text="B/C", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="mln PLN", row=1, col=1, secondary_y=True)
fig.update_yaxes(title_text="bp", row=2, col=1)

fig.update_layout(
    title="Statystyki polskich aukcji obligacji skarbowych (AS+AU)",
    template="plotly_white", height=700,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.10),
)
fig.show()

### 4b. B/C i tail per typ kuponu (I / OS / Z)

- **I** — inflation-linked (Oprocentowanie='I', np. IZ)
- **OS** — zero-coupon + stałe łącznie (O+S, np. OK + PS + DS + WS)
- **Z** — zmienne (WZ, NZ)

In [ ]:
df4b = fetch_view(
    "v_auction_by_coupon_bucket",
    "?type_op=in.(AS,AU)&select=*&order=auction_date.asc,coupon_bucket.asc",
)
df4b["auction_date"] = pd.to_datetime(df4b["auction_date"])
for c in ["bid_to_cover", "w_yield_avg", "w_tail_bp", "w_concession_bp", "total_sold_mln"]:
    df4b[c] = pd.to_numeric(df4b[c], errors="coerce")

print(f"Rows: {len(df4b)},  buckets: {sorted(df4b.coupon_bucket.unique())}")

BUCKET_COLORS = {"I": "darkviolet", "OS": "navy", "Z": "darkorange"}

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                    subplot_titles=("Bid-to-cover", "Tail (bp)"))

for bucket in sorted(df4b.coupon_bucket.unique()):
    sub = df4b[df4b.coupon_bucket == bucket].sort_values("auction_date")
    color = BUCKET_COLORS.get(bucket, "grey")
    fig.add_trace(go.Scatter(x=sub["auction_date"], y=sub["bid_to_cover"],
                             name=bucket, legendgroup=bucket, mode="lines+markers",
                             line=dict(color=color, width=1.5), marker=dict(size=4)),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=sub["auction_date"], y=sub["w_tail_bp"],
                             name=bucket, legendgroup=bucket, showlegend=False,
                             mode="lines+markers",
                             line=dict(color=color, width=1.5), marker=dict(size=4)),
                  row=2, col=1)

fig.update_layout(
    title="Metryki aukcyjne per kubełek kuponu",
    template="plotly_white", height=700,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.10),
)
fig.update_yaxes(title_text="B/C", row=1, col=1)
fig.update_yaxes(title_text="bp", row=2, col=1)
fig.show()

### 4c. B/C, tail i yield per tenor (LataDoWykupu z MF)

Po każdym tenorze osobno (2/5/10/20/30 lat itd.). Pokazane tylko tenory które pojawiły się ≥20 razy. Trzeci panel to średni ważony yield = praktycznie krzywą rentowności aukcyjnej w czasie.

In [ ]:
df4c = fetch_view(
    "v_auction_by_tenor",
    "?type_op=in.(AS,AU)&select=*&order=auction_date.asc,years_to_maturity.asc",
)
df4c["auction_date"] = pd.to_datetime(df4c["auction_date"])
for c in ["bid_to_cover", "w_yield_avg", "w_tail_bp", "w_concession_bp", "total_sold_mln"]:
    df4c[c] = pd.to_numeric(df4c[c], errors="coerce")

common = df4c["years_to_maturity"].value_counts()
keep_tenors = sorted([t for t in common.index if common[t] >= 20])
print(f"Rows: {len(df4c)},  tenors shown (>=20 wystąpień): {keep_tenors}")

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=("Bid-to-cover per tenor",
                                    "Tail (bp) per tenor",
                                    "Średnia ważona rentowność (%) per tenor"))

palette = px.colors.qualitative.Plotly + px.colors.qualitative.Set2
color_map = {t: palette[i % len(palette)] for i, t in enumerate(keep_tenors)}

for tenor in keep_tenors:
    sub = df4c[df4c.years_to_maturity == tenor].sort_values("auction_date")
    name = f"{int(tenor)}Y"
    color = color_map[tenor]
    fig.add_trace(go.Scatter(x=sub["auction_date"], y=sub["bid_to_cover"],
                             name=name, legendgroup=name, mode="lines+markers",
                             line=dict(color=color, width=1.3), marker=dict(size=3)),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=sub["auction_date"], y=sub["w_tail_bp"],
                             name=name, legendgroup=name, showlegend=False,
                             mode="lines+markers",
                             line=dict(color=color, width=1.3), marker=dict(size=3)),
                  row=2, col=1)
    fig.add_trace(go.Scatter(x=sub["auction_date"], y=sub["w_yield_avg"],
                             name=name, legendgroup=name, showlegend=False,
                             mode="lines+markers",
                             line=dict(color=color, width=1.3), marker=dict(size=3)),
                  row=3, col=1)

fig.update_layout(
    title="Metryki aukcyjne per LataDoWykupu",
    template="plotly_white", height=900,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.08),
)
fig.update_yaxes(title_text="B/C", row=1, col=1)
fig.update_yaxes(title_text="bp", row=2, col=1)
fig.update_yaxes(title_text="%", row=3, col=1)
fig.show()

### 4d. Ostatnia aukcja — szczegóły per seria

Tabela wszystkich serii sprzedanych w najnowszej dacie. `concession_bp` > 0 = aukcja droga vs rynek wtórny, < 0 = "through" (taniej niż rynek).

In [ ]:
df4d_recent = fetch_view(
    "v_recent_auctions",
    "?type_op=in.(AS,AU)&select=*&order=auction_date.desc&limit=50",
)
df4d_recent["auction_date"] = pd.to_datetime(df4d_recent["auction_date"]).dt.date

latest_date = df4d_recent["auction_date"].max()
df_last = df4d_recent[df4d_recent["auction_date"] == latest_date].copy()
for c in ["offer_max_mln", "demand_total_mln", "sold_total_mln",
          "bid_to_cover", "tail_yield_bp", "yield_avg",
          "concession_bp", "nc_share_demand"]:
    if c in df_last.columns:
        df_last[c] = pd.to_numeric(df_last[c], errors="coerce")

total_sold = df_last["sold_total_mln"].sum()
total_demand = df_last["demand_total_mln"].sum()
overall_bc = total_demand / total_sold if total_sold else float("nan")

print(f"Ostatnia aukcja: {latest_date}  ({len(df_last)} serii)")
print(f"Łączny sold:    {total_sold:>10,.1f} mln PLN")
print(f"Łączny demand:  {total_demand:>10,.1f} mln PLN")
print(f"B/C całej aukcji: {overall_bc:.2f}")

cols = ["seria", "years_to_maturity", "coupon_kind", "offer_max_mln",
        "demand_total_mln", "sold_total_mln", "bid_to_cover",
        "yield_avg", "tail_yield_bp", "concession_bp", "nc_share_demand"]
view = df_last[cols].rename(columns={
    "years_to_maturity": "yrs",
    "coupon_kind": "kind",
    "offer_max_mln": "offer",
    "demand_total_mln": "demand",
    "sold_total_mln": "sold",
    "bid_to_cover": "B/C",
    "yield_avg": "yld_avg %",
    "tail_yield_bp": "tail bp",
    "concession_bp": "conc bp",
    "nc_share_demand": "NK %",
})
view["NK %"] = view["NK %"] * 100
view.style.format({
    "offer": "{:,.0f}", "demand": "{:,.0f}", "sold": "{:,.0f}",
    "B/C": "{:.2f}", "yld_avg %": "{:.3f}", "tail bp": "{:.1f}",
    "conc bp": "{:+.1f}", "NK %": "{:.1f}",
}, na_rep="—")

---

**Tip:** wykresy plotly są interaktywne — najedź myszą żeby zobaczyć wartości, zaznacz prostokąt żeby przybliżyć, podwójny klik żeby zresetować. Trace w legendzie można wyłączać klikiem.